# Predicting Language Model Memorization from Tokenizer Artifacts

**Setup:** Runtime > Change runtime type > **T4 GPU** > Save

Three configs available:

| Config | Corpus | Candidates | Model | Time |
|--------|--------|------------|-------|------|
| `colab_nuke` | Synthetic 5M chars | 30 canaries | 2.5M params | ~10 min |
| `colab_real` | WikiText-103 20M chars | 1000 (100+900) | 4.3M params | ~45 min |
| `colab_max` | WikiText-103 50M chars | 5000 (500+4500) | 28M params | ~2.5-3 hr |

In [ ]:
# Clone repo
!git clone https://github.com/louatiMahmoudAziz/Predicting-Language-Model-Memorization-from-Tokenizer-Artifacts.git
%cd Predicting-Language-Model-Memorization-from-Tokenizer-Artifacts

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q datasets

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Option A: Smoke test (`colab_nuke`, ~10 min)
Small synthetic corpus, 30 canaries. Verifies the pipeline works.

In [ ]:
# Option A: smoke test
!python scripts/gen_nuke_data.py
!python -m src.run_pipeline --config configs/colab_nuke.yaml

---
## Option B: Realistic (`colab_real`, ~45 min)
WikiText-103, 1000 candidates (100 canaries + 900 benign), 4.3M param model.

In [ ]:
# Option B: realistic
!python scripts/gen_real_data.py
!python -m src.run_pipeline --config configs/colab_real.yaml

---
## Option C: Maximum scale (`colab_max`, ~2.5-3 hr)
WikiText-103 50M chars, 5000 candidates (500 diverse canaries + 4500 benign),
28M param model, 16K BPE vocab, seq_len=1024.

Canary diversity: synthetic secrets, PII patterns, near-natural Wikipedia text.
All three quantile thresholds (5%, 1%, 0.1%) are resolvable.

In [ ]:
# Option C: maximum scale
!python scripts/gen_max_data.py
!python -m src.run_pipeline --config configs/colab_max.yaml

---
## Inspect results

In [ ]:
import pandas as pd
import json

# Set to whichever config you ran
RUN_ID = "colab_max"  # or "colab_real" or "colab_nuke"

# Pipeline summary
manifest = json.load(open(f"results/{RUN_ID}/{RUN_ID}_pipeline.json"))
print("=== PIPELINE STEPS ===")
total_s = 0
for s in manifest["steps"]:
    t = s.get("elapsed_s", 0) or 0
    total_s += t
    print(f"  {s['step']:02d}  {s['name']:<22}  {s['status']:<8}  {t:7.1f}s")
print(f"  TOTAL: {total_s:.0f}s  ({total_s/60:.1f} min)")

In [ ]:
# Delta BPC distribution
labels = pd.read_parquet(f"labels/{RUN_ID}_labels.parquet")
valid = labels[labels["valid_label"] == True].copy()

is_canary = valid["candidate_id"].str.startswith("canary_")
print(f"=== DELTA BPC (valid: {len(valid)} / {len(labels)}) ===")
print(f"  Canaries ({is_canary.sum():>5d}):  "
      f"mean={valid.loc[is_canary, 'delta_bpc'].mean():.3f}  "
      f"min={valid.loc[is_canary, 'delta_bpc'].min():.3f}  "
      f"max={valid.loc[is_canary, 'delta_bpc'].max():.3f}")
if (~is_canary).any():
    print(f"  Benign  ({(~is_canary).sum():>5d}):  "
          f"mean={valid.loc[~is_canary, 'delta_bpc'].mean():.3f}  "
          f"min={valid.loc[~is_canary, 'delta_bpc'].min():.3f}  "
          f"max={valid.loc[~is_canary, 'delta_bpc'].max():.3f}")

# Threshold labels
for col in ["label_top_5pct", "label_top_1pct", "label_top_0_1pct"]:
    if col in valid.columns:
        n_pos = int(valid[col].sum()) if valid[col].notna().any() else 0
        n_na = int(valid[col].isna().sum())
        status = f"{n_pos} positives" if n_na == 0 else f"NaN ({n_na} rows)"
        print(f"  {col}: {status}")

In [ ]:
# Evaluation metrics
comp = pd.read_parquet(f"results/{RUN_ID}/eval/comparison.parquet")
display_cols = [c for c in ["task", "model_name", "feature_subset", "n_test", "n_pos",
                            "auroc", "auprc", "reg_pearson_r", "reg_spearman_rho"]
                if c in comp.columns]

# Show non-NaN AUROC rows first
has_auroc = comp["auroc"].notna() if "auroc" in comp.columns else comp.index < 0
print("=== METRICS (rows with valid AUROC) ===")
display(comp.loc[has_auroc, display_cols].sort_values("auroc", ascending=False))

---
## Save results to Google Drive (optional)
Uncomment and run to persist results after session ends.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# dst = f"/content/drive/MyDrive/{RUN_ID}_results"
# shutil.copytree(f"results/{RUN_ID}", dst, dirs_exist_ok=True)
# shutil.copytree("labels", f"{dst}/labels", dirs_exist_ok=True)
# shutil.copytree("features", f"{dst}/features", dirs_exist_ok=True)
# print(f"Results saved to Google Drive: {dst}")